<a href="https://colab.research.google.com/github/vedang-work/Resume-Matching/blob/main/Redrob_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import re
from collections import defaultdict

In [ ]:
# SKILL_ALIASES
SKILL_ALIASES = {
    # Languages
    "python": "python",
    "pyhton": "python",
    "java": "java",
    "javascript": "javascript",
    "javascrpit": "javascript",
    "js": "javascript",
    "typescript": "typescript",
    "typescrpit": "typescript",
    "c++": "cpp",
    "cpp": "cpp",
    "r": "r",
    "kotlin": "kotlin",
    # ML / Data
    "machinelearning": "machine_learning",
    "machine learning": "machine_learning",
    "ml": "machine_learning",
    "sklearn": "machine_learning",
    "deeplearning": "deep_learning",
    "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow",
    "pytorch": "pytorch",
    "keras": "keras",
    "nlp": "nlp",
    "bert": "bert",
    "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics",
    "stats": "statistics",
    "regression": "regression",
    "clustering": "clustering",
    "data-viz": "data_visualization",
    "data visualization": "data_visualization",
    "data viz": "data_visualization",
    "matplotlib": "data_visualization",
    "tableau": "data_visualization",
    "power-bi": "data_visualization",
    "power bi": "data_visualization",
    "powerbi": "data_visualization",
    "pandas": "pandas",
    "numpy": "numpy",
    # Web — Frontend
    "react": "react",
    "reacts": "react",
    "reactjs": "react",
    "vue": "vue",
    "vue.js": "vue",
    "vuejs": "vue",
    "redux": "redux",
    "tailwind": "tailwind",
    "html/css": "html_css",
    "html css": "html_css",
    "html": "html_css",
    "css": "html_css",
    "jest": "jest",
    "graphql": "graphql",
    # Web — Backend
    "node.js": "nodejs",
    "nodejs": "nodejs",
    "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot",
    "springboot": "spring_boot",
    "rest api": "rest_api",
    "rest": "rest_api",
    "restapi": "rest_api",
    "microservices": "microservices",
    # Databases
    "sql": "sql",
    "mysql": "mysql",
    "mysq": "mysql",
    "postgresql": "postgresql",
    "postgres": "postgresql",
    "mongodb": "mongodb",
    "redis": "redis",
    # DevOps / Cloud
    "docker": "docker",
    "kubernetes": "kubernetes",
    "kubernates": "kubernetes",
    "k8s": "kubernetes",
    "ci/cd": "ci_cd",
    "cicd": "ci_cd",
    "ci cd": "ci_cd",
    "aws": "aws",
    # Mobile
    "android": "android",
    "firebase": "firebase",
    # CS Fundamentals
    "algorithms": "algorithms",
    "algoritms": "algorithms",
    "data structure": "data_structures",
    "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    # Design
    "ui/ux": "ui_ux",
    "ui ux": "ui_ux",
    "figma": "figma",
}

In [ ]:
# Resume Dataset
resumes = [
    {"id": "01", "name": "Arjun Sharma", "raw_skills": "Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning"},
    {"id": "02", "name": "Priya Nair", "raw_skills": "JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS"},
    {"id": "03", "name": "Rahul Gupta", "raw_skills": "Java, Spring Boot, MySql, Microservices, Docker, kubernates"},
    {"id": "04", "name": "Sneha Patel", "raw_skills": "Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib"},
    {"id": "05", "name": "Vikram Singh", "raw_skills": "C++, Algoritms, Data Structure, competitive programming, python"},
    {"id": "06", "name": "Ananya Krishnan", "raw_skills": "javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD"},
    {"id": "07", "name": "Karan Mehta", "raw_skills": "Python, Sklearn, XGboost, feature engineering, SQL, tableau"},
    {"id": "08", "name": "Deepika Rao", "raw_skills": "Java, Android, Kotlin, Firebase, REST, UI/UX, figma"},
    {"id": "09", "name": "Aditya Kumar", "raw_skills": "Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest"},
    {"id": "10", "name": "Meera Iyer", "raw_skills": "python, R, statistics, ML, regression, clustering, Power-BI"},
]

In [ ]:
# Job Description Dataset
job_descriptions = [
    {"id": "JD-1", "company": "Kakao", "role": "ML Engineer", "required_skills": ["Python", "Machine Learning", "Deep Learning", "TensorFlow", "PyTorch", "SQL", "Data Visualization"], "preferred_skills": ["NLP", "BERT", "Feature Engineering", "Statistics"]},
    {"id": "JD-2", "company": "Naver", "role": "Backend Engineer", "required_skills": ["Java", "Spring Boot", "MySQL", "PostgreSQL", "Microservices", "Docker", "Kubernetes"], "preferred_skills": ["REST API", "CI/CD", "Redis"]},
    {"id": "JD-3", "company": "Line", "role": "Frontend Engineer", "required_skills": ["JavaScript", "React", "Vue", "TypeScript", "REST API", "HTML/CSS"], "preferred_skills": ["Node.js", "GraphQL", "Redux", "Jest", "AWS"]},
]

In [ ]:
def normalize_skills(raw_skills):
    skills = []
    for skill in raw_skills.split(','):
        skill = skill.strip().lower()
        if skill in SKILL_ALIASES:
            skills.append(SKILL_ALIASES[skill])
        else:
            for alias, skill_name in SKILL_ALIASES.items():
                if alias in skill:
                    skills.append(skill_name)
                    break
    return list(set(skills))

In [ ]:
def compute_tf_idf(resumes):
    vocabulary = set()
    for resume in resumes:
        vocabulary.update(resume['skills'])
    vocabulary = sorted(list(vocabulary))
    tf_idf = []
    for resume in resumes:
        tf_idf_vector = [0] * len(vocabulary)
        for skill in resume['skills']:
            tf_idf_vector[vocabulary.index(skill)] = 1 / len(resume['skills'])
        idf = {}
        for skill in vocabulary:
            idf[skill] = math.log(10 / sum(1 for r in resumes if skill in r['skills']))
        for i, skill in enumerate(vocabulary):
            tf_idf_vector[i] *= idf[skill]
        tf_idf.append(tf_idf_vector)
    return tf_idf, vocabulary

In [ ]:
def compute_jd_vector(job_description, vocabulary):
    jd_vector = [0] * len(vocabulary)
    for skill in job_description['required_skills'] + job_description['preferred_skills']:
        skill = skill.lower()
        if skill in SKILL_ALIASES:
            skill = SKILL_ALIASES[skill]
        if skill in vocabulary:
            jd_vector[vocabulary.index(skill)] = 1
    return jd_vector

In [ ]:
def compute_cosine_similarity(tf_idf_vector, jd_vector):
    dot_product = sum(a * b for a, b in zip(tf_idf_vector, jd_vector))
    magnitude_a = math.sqrt(sum(a ** 2 for a in tf_idf_vector))
    magnitude_b = math.sqrt(sum(a ** 2 for a in jd_vector))
    return dot_product / (magnitude_a * magnitude_b)


In [ ]:
def rank_candidates(resumes, job_description):
    tf_idf, vocabulary = compute_tf_idf(resumes)
    jd_vector = compute_jd_vector(job_description, vocabulary)
    scores = []
    for i, resume in enumerate(resumes):
        score = compute_cosine_similarity(tf_idf[i], jd_vector)
        scores.append((resume['name'], score))
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores[:3]

In [ ]:
def main():
    for resume in resumes:
        resume['skills'] = normalize_skills(resume['raw_skills'])
    for job_description in job_descriptions:
        scores = rank_candidates(resumes, job_description)
        print(f"{job_description['id']} — {job_description['company']} ({job_description['role']})")
        for name, score in scores:
            print(f"{name}({score:.2f})")
        print()

if __name__ == "__main__":
    main()